# Phase 5 — 이미지 처리 기본기 (예제)

전처리 기법을 순서대로 적용해 '특정 층을 깔끔하게 분리'하는 흐름을 익힙니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

그래프 라벨에 한글을 쓸 때 깨지지 않도록, 설치된 한글 폰트를 찾아 적용합니다. (없으면 라벨은 영어로 표기)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; print('한글 폰트 적용:', _n); break
else:
    print('한글 폰트를 찾지 못했습니다 — 라벨은 영어로 표기합니다.')
plt.rcParams['axes.unicode_minus'] = False

## 1. 예제 이미지 준비
실제 TEM 파일이 없어도 실행되도록, 저대비 + 노이즈가 있는 TEM 유사 이미지를 생성합니다. 실제 파일이 있으면 `img = cv2.imread('경로.png', cv2.IMREAD_GRAYSCALE)` 로 바꾸면 됩니다.

In [ ]:
import numpy as np, cv2
np.random.seed(7)

H, W = 240, 360
base = np.zeros((H, W))
base[:70]=95; base[70:100]=150; base[100:120]=60; base[120:175]=135; base[175:]=100
base = base*0.55 + 70                      # 대비를 낮춤
img = np.clip(base + np.random.normal(0,12,(H,W)), 0, 255).astype(np.uint8)

plt.imshow(img, cmap='gray', vmin=0, vmax=255); plt.title('sample (low contrast)')
plt.axis('off'); plt.show()
print('shape:', img.shape, 'dtype:', img.dtype)

## 2. 밝기·대비 보정: normalization
픽셀값 범위를 0~255 로 늘려 대비를 높입니다.

In [ ]:
norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('원본 (저대비)'); ax[0].axis('off')
ax[1].imshow(norm, cmap='gray'); ax[1].set_title('정규화 후'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 3. 히스토그램 평활화 · CLAHE
equalizeHist 는 전역, CLAHE 는 영역별로 대비를 높입니다. 밝기 편차가 큰 TEM 에는 CLAHE 가 유리합니다.

In [ ]:
eq = cv2.equalizeHist(img)
clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8)).apply(img)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(eq, cmap='gray'); ax[0].set_title('equalizeHist'); ax[0].axis('off')
ax[1].imshow(clahe, cmap='gray'); ax[1].set_title('CLAHE'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 4. 노이즈 제거
Gaussian(부드럽게), Median(점잡음에 강함), Bilateral(경계 보존) 중 상황에 맞게 선택합니다.

In [ ]:
g   = cv2.GaussianBlur(img, (5,5), 0)
med = cv2.medianBlur(img, 5)
bil = cv2.bilateralFilter(img, 9, 50, 50)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(g, cmap='gray'); ax[0].set_title('GaussianBlur'); ax[0].axis('off')
ax[1].imshow(med, cmap='gray'); ax[1].set_title('medianBlur'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 5. 이진화: 층/배경 나누기
Otsu 는 임계값을 자동으로 정합니다. Adaptive 는 영역별로 다른 임계값을 사용합니다.

In [ ]:
den = cv2.GaussianBlur(img, (5,5), 0)   # 보통 노이즈 제거 후 이진화
thr, binary = cv2.threshold(den, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print('Otsu 가 정한 임계값 :', thr)
adap = cv2.adaptiveThreshold(den, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                             cv2.THRESH_BINARY, 31, 5)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(binary, cmap='gray'); ax[0].set_title('Otsu'); ax[0].axis('off')
ax[1].imshow(adap, cmap='gray'); ax[1].set_title('Adaptive'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 6. 모폴로지: 이진 이미지 다듬기
열림(opening)은 작은 흰 점을 제거, 닫힘(closing)은 작은 검은 구멍을 메웁니다.

In [ ]:
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k)
closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(opening, cmap='gray'); ax[0].set_title('열림(opening)'); ax[0].axis('off')
ax[1].imshow(closing, cmap='gray'); ax[1].set_title('닫힘(closing)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 7. 엣지 검출
Canny 는 깔끔한 엣지 선을 줍니다. 층 경계 위치 파악에 활용합니다.

In [ ]:
edge = cv2.Canny(den, 40, 120)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(den, cmap='gray'); ax[0].set_title('입력'); ax[0].axis('off')
ax[1].imshow(edge, cmap='gray'); ax[1].set_title('Canny 엣지'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 8. 윤곽선(contour) 검출
이진 영역의 외곽선을 찾고, 면적·둘레를 측정에 활용합니다. (면적이 큰 윤곽선부터 정렬)

In [ ]:
cnts, _ = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
print('찾은 윤곽선 개수 :', len(cnts))
if cnts:
    print('가장 큰 윤곽선 면적(px) :', cv2.contourArea(cnts[0]))

ov = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
cv2.drawContours(ov, cnts, -1, (13,148,136), 2)
plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)); plt.title('contours'); plt.axis('off'); plt.show()

## 8-1. 윤곽선 그리기와 컬러 오버레이 (drawContours)
찾은 윤곽선을 `drawContours` 로 그립니다. **그레이스케일은 1채널이라 색이 표현되지 않으므로**, `cv2.cvtColor(gray, COLOR_GRAY2BGR)` 로 3채널로 바꾼 뒤 컬러 윤곽선을 얹습니다. (표시는 matplotlib 기준 RGB 로 변환 — Phase 2 참고)

In [ ]:
cnts, _ = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 1채널 그레이 -> 3채널(BGR) : 색을 쓰려면 필수
vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
cv2.drawContours(vis, cnts, -1, (136,148,13), 2)      # teal (BGR)
big = max(cnts, key=cv2.contourArea)
cv2.drawContours(vis, [big], -1, (60,60,220), 2)      # 큰 윤곽선 빨강 (BGR)

# matplotlib 은 RGB 로 표시하므로 BGR -> RGB 변환 후 표시
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('color contours on grayscale'); plt.axis('off'); plt.show()

**반투명 채우기 (addWeighted)** — 영역을 색으로 채운 사본을 만든 뒤 원본과 섞습니다.

In [ ]:
fill = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
layer = fill.copy()
cv2.drawContours(layer, cnts, -1, (136,148,13), cv2.FILLED)   # 안을 채움
blend = cv2.addWeighted(layer, 0.35, fill, 0.65, 0)           # 35% 반투명
cv2.drawContours(blend, cnts, -1, (136,148,13), 2)            # 테두리도
plt.imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB))
plt.title('semi-transparent fill'); plt.axis('off'); plt.show()

## 9. 값 기반 마스크 생성
특정 밝기 구간의 픽셀만 골라 마스크를 만듭니다. 원하는 층만 분리할 수 있습니다. (컬러는 inRange 에 (B,G,R) 하한·상한)

In [ ]:
mask = cv2.inRange(g, 140, 255)        # 밝기 140~255 인 픽셀
print('선택된 픽셀 수 :', cv2.countNonZero(mask))

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(8,3))
ax[0].imshow(g, cmap='gray'); ax[0].set_title('입력'); ax[0].axis('off')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('밝은 층 마스크'); ax[1].axis('off')
plt.tight_layout(); plt.show()

---
예제 실행을 마친 후 `05_practice.ipynb` 의 문제를 해결하십시오.